<div align="center">

# Fine-Tuning Gemma 4 E4B IT for Agentic Reasoning + Tool Use

Language: English

Source: [TINYCUA](https://github.com/VJyzCELERY/TINYCUA)

Status: Experimental

<br>

</div>

---

## Model & Dataset

- **Base Model**: [unsloth/gemma-4-E4B-it](https://huggingface.co/unsloth/gemma-4-E4B-it) (4.5B effective / 8B total params)
- **Base**: [google/gemma-4-E4B](https://huggingface.co/google/gemma-4-E4B)
- **Fine-tuning Method**: QLoRA (NF4 + Double Quantization) via Unsloth
- **Dataset**: [lambda/hermes-agent-reasoning-traces](https://huggingface.co/datasets/lambda/hermes-agent-reasoning-traces) (kimi config, ~7.6K samples)
- **Evaluation**: ToolBench tool-call accuracy
- **Tracking**: Weights & Biases

---

## Pipeline Overview

1. Configure API keys (HF, W&B)
2. Install dependencies
3. Detect GPU
4. Load model with 4-bit QLoRA
5. Apply LoRA adapters (r=64, RSLoRA)
6. Load and preprocess hermes-agent-reasoning-traces
7. Apply Gemma 4 chat template
8. Train with SFTTrainer (test run: 20 steps, full: 1 epoch)
9. Evaluate on ToolBench subset (inference-based)
10. Push LoRA adapter to HuggingFace Hub
11. Save merged 16-bit model + GGUF export (optional)


## Before You Start: Required API Keys

Store these in **Colab Secrets** (key icon in the left sidebar).

### 1. HF_TOKEN
- Hugging Face token with write access to push LoRA adapters.
- Create at: https://huggingface.co/settings/tokens

### 2. WANDB_API_KEY
- Weights & Biases API key for experiment tracking.
- Get at: https://wandb.ai/authorize

### 3. How to store in Colab Secrets
1. Click the key icon in the left Colab toolbar.
2. Add HF_TOKEN and WANDB_API_KEY with their values.
3. Enable "Notebook access" toggle for each.


In [ ]:
# ============================================================
# API KEY SETUP + GDRIVE MOUNT
# ============================================================
from google.colab import userdata
import os

# Weights & Biases
try:
    wandb_key = userdata.get("WANDB_API_KEY")
    if wandb_key:
        import wandb
        wandb.login(key=wandb_key)
        print("Logged in to W&B successfully!")
except userdata.SecretNotFoundError:
    print("WANDB_API_KEY not found. W&B logging will be disabled.")
    wandb_key = None

# Hugging Face (required for downloading Gemma 4)
try:
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print("Logged in to HuggingFace!")
except userdata.SecretNotFoundError:
    print("HF_TOKEN not found. HF push will be disabled.")
    hf_token = None

# Mount Google Drive for checkpoint persistence across Colab sessions
from google.colab import drive
drive.mount("/content/drive")

# Output directory (on GDrive so checkpoints survive session timeout)
output_directory = "/content/drive/MyDrive/tinycua-finetune/checkpoints"
os.makedirs(output_directory, exist_ok=True)
print(f"Checkpoints will be saved to: {output_directory}")


In [ ]:
%%capture
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
import os, re, torch

if "COLAB_" in "".join(os.environ.keys()):
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10":"0.0.34","2.9":"0.0.33.post1","2.8":"0.0.32.post2"}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
else:
    !pip install unsloth
!pip install "transformers>=5.3.0" "trl>=0.22.2"


In [ ]:
# ============================================================
# GPU DETECTION
# ============================================================
print("=" * 50)
print("GPU Configuration")
print("=" * 50)
import torch
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs: {num_gpus}")
    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        vram_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i}: {gpu_name} ({vram_gb:.1f}GB)")
    print("Using CUDA for training")
else:
    print("WARNING: No GPU detected - using CPU (will be very slow)")
print("=" * 50)


## Load Model

Load Gemma 4 E4B IT with 4-bit NF4 quantization via Unsloth.

In [ ]:
# ============================================================
# LOAD MODEL AND TOKENIZER
# ============================================================
from unsloth import FastModel
import torch

MODEL_NAME = "unsloth/gemma-4-E4B-it"
MAX_SEQ_LENGTH = 8192

print("Loading model...")
model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    # load_in_8bit = False,
    full_finetuning = False,
    device_map = "auto",
    dtype = None,  # None for auto detection
)
print("Model loaded successfully!")
if torch.cuda.is_available():
    print(f"Model device: {next(model.parameters()).device}")
print(f"Tokenizer vocab size: {len(tokenizer.tokenizer)}")
print("Setup complete! Ready for LoRA configuration.")


## LoRA Configuration

Apply QLoRA adapters to attention and MLP layers with RSLoRA for stable training.

In [ ]:
# ============================================================
# LORA CONFIGURATION (Gemma 4 API)
# ============================================================
RANDOM_SEED = 3407
LEARNING_RATE = 2e-4
LORA_RANK = 64

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # Text-only fine-tuning
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = LORA_RANK,
    lora_alpha = LORA_RANK,
    lora_dropout = 0,
    bias = "none",
    random_state = RANDOM_SEED,
    use_rslora = True,
)
print("LoRA adapter configured with QLoRA optimization (RSLoRA enabled)")


## Dataset: lambda/hermes-agent-reasoning-traces

Multi-turn tool-calling trajectories with chain-of-thought reasoning (<think> blocks) and actual tool execution results. Generated by the [Hermes Agent](https://github.com/nousresearch/hermes-agent) harness.

**Config: kimi** - 7,646 samples generated by Moonshot AI Kimi-K2.5

### Dataset Statistics

| Metric | Value |
|--------|-------|
| Samples | 7,646 |
| Total turns | 185,798 |
| Total tool calls | 106,222 |
| Avg turns per sample | 24.3 |
| Avg tool calls per sample | 13.9 |
| Avg <think> depth (words) | 414 |

### Categories
- Terminal & Coding (26%)
- Agent Tools (19%)
- Repository Tasks (15%)
- Browser Automation (14%)
- Multi-Tool, File Operations, Scheduling, Planning, Conversational

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================
from datasets import load_dataset
import json

DATASET_CONFIG = "kimi"

print(f"Loading hermes-agent-reasoning-traces ({DATASET_CONFIG} config)...")
dataset = load_dataset("lambda/hermes-agent-reasoning-traces", DATASET_CONFIG, split="train")

print(f"
Dataset loaded: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")

sample = dataset[0]
print(f"
First example:")
print(f"  Category: {sample['category']}")
print(f"  Subcategory: {sample['subcategory']}")
print(f"  Conversations: {len(sample['conversations'])} turns")


In [ ]:
# ============================================================
# DATASET PREPROCESSING
# ============================================================
# Convert ShareGPT format to standard messages

ROLE_MAPPING = {
    "system": "system",
    "human": "user",
    "gpt": "assistant",
    "tool": "tool",
}

def convert_to_messages(example):
    conversations = example["conversations"]
    messages = []
    for turn in conversations:
        role = ROLE_MAPPING.get(turn.get("from", ""))
        content = turn.get("value", "")
        if not role or not content:
            continue
        # Merge tool responses into preceding assistant, then drop
        if role == "tool":
            if messages and messages[-1]["role"] == "assistant":
                messages[-1]["content"] += f"\n\n[Tool Result]\n{content}\n[/Tool Result]"
            continue
        # Merge consecutive same-role messages (handles tool-calling
        # chains like assistant->assistant, and user->user)
        if messages and messages[-1]["role"] == role:
            messages[-1]["content"] += f"\n\n{content}"
            continue
        messages.append({"role": role, "content": content})
    # Final validation: must have >=2 messages, start with
    # user/system, end with assistant, no non-standard roles
    if len(messages) < 2:
        return {"messages": None}
    if messages[-1]["role"] != "assistant":
        return {"messages": None}
    if messages[0]["role"] not in ("user", "system"):
        return {"messages": None}
    return {"messages": messages}

print("Converting ShareGPT format to standard messages...")
processed = dataset.map(
    convert_to_messages,
    remove_columns=dataset.column_names,
    num_proc=4,
    load_from_cache_file=False,
)
processed = processed.filter(lambda x: x["messages"] is not None)
print(f"After conversion: {len(processed)} valid samples")


In [ ]:
# ============================================================
# APPLY GEMMA 4 CHAT TEMPLATE
# ============================================================
from unsloth.chat_templates import get_chat_template

try:
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "gemma-4",
    )
    print("Using Unsloth gemma4 chat template")
except Exception:
    print("Unsloth gemma4 template not available, using default")

def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

print("Applying chat template...")
dataset_final = processed.map(
    apply_chat_template,
    remove_columns=processed.column_names,
    num_proc=4,
)
print(f"Final dataset: {len(dataset_final)} samples")

# Filter by token length
def filter_by_length(example):
    tokens = tokenizer.tokenizer(example["text"], add_special_tokens=False)
    return len(tokens["input_ids"]) <= MAX_SEQ_LENGTH

before_count = len(dataset_final)
dataset_final = dataset_final.filter(filter_by_length, num_proc=4)
after_count = len(dataset_final)
print(f"Filtered: {before_count - after_count}/{before_count} removed ({100*(before_count-after_count)/before_count:.1f}%)")


In [ ]:
# ============================================================
# TRAINING CONFIGURATION - TEST RUN (20 steps)
# ============================================================
if len(dataset_final) == 0:
    raise ValueError("Dataset is empty after filtering. Try increasing MAX_SEQ_LENGTH.")

from trl import SFTTrainer, SFTConfig

WANDB_PROJECT = "gemma4-e4b-hermes-agent-reasoning"

if wandb_key:
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_final,
    eval_dataset = None,
    max_seq_length = MAX_SEQ_LENGTH,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.04,
        max_steps = 20,
        # num_train_epochs = 1,
        learning_rate = LEARNING_RATE,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = RANDOM_SEED,
        save_steps = 10,
        save_total_limit = 3,
        save_strategy = "steps",
        report_to = "wandb" if wandb_key else "none",
        output_dir = output_directory,
    ),
)
print(f"Trainer configured:")
print(f"  - Batch size: 2 (per device)")
print(f"  - Gradient accumulation: 8")
print(f"  - Effective batch: 16")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - Max steps: 20 (test run)")
print(f"  - Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  - W&B project: {WANDB_PROJECT}")


## Training - Test Run

Trains for **20 steps** to validate the pipeline.

For full training: change max_steps to num_train_epochs = 1 in the cell above.

In [ ]:
import glob, os
print("=" * 50)
print("Starting training...")
print("=" * 50)

checkpoint_dirs = sorted(glob.glob(os.path.join(output_directory, "checkpoint-*")))
resume_from = checkpoint_dirs[-1] if checkpoint_dirs else None
if resume_from:
    print(f"Resuming from checkpoint: {resume_from}")


trainer.train(resume_from_checkpoint=resume_from)


print("=" * 50)
print("Training complete!")
print("=" * 50)


## Full Training (1 Epoch)

Run **after** the test run succeeds (~4-6 hrs on T4).
Checkpoints auto-save every 50 steps. If Colab times out,
just re-run this cell. It **auto-resumes** from the latest
checkpoint on Google Drive.


In [ ]:
# ============================================================
# FULL TRAINING - 1 EPOCH (auto-resumes if interrupted)
# ============================================================
import glob, os
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_final,
    eval_dataset = None,
    max_seq_length = MAX_SEQ_LENGTH,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.04,
        num_train_epochs = 1,
        learning_rate = LEARNING_RATE,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = RANDOM_SEED,
        save_steps = 50,
        save_total_limit = 5,
        save_strategy = "steps",
        report_to = "wandb" if wandb_key else "none",
        output_dir = output_directory,
    ),
)

print("=" * 50)
print("Starting full training (1 epoch)...")
print("=" * 50)

checkpoint_dirs = sorted(glob.glob(os.path.join(output_directory, "checkpoint-*")))
resume_from = checkpoint_dirs[-1] if checkpoint_dirs else None
if resume_from:
    print(f"Resuming from checkpoint: {resume_from}")

trainer.train(resume_from_checkpoint=resume_from)

print("=" * 50)
print("Full training complete!")
print("=" * 50)


## ToolBench Evaluation (Inference-Based)

Evaluates the fine-tuned model by running inference on a ToolBench subset.
Measures tool-call exact-match accuracy.

**What we measure:**
- Tool name exact-match accuracy
- Argument JSON validity
- Overall task completion rate

In [ ]:
# ============================================================
# TOOLBENCH EVALUATION HARNESS (inference-based)
# ============================================================
import json, re
from tqdm import tqdm

def extract_tool_calls(text):
    tool_calls = []
    pattern = r"<tool_call>\s*(\w+)\s*\(([^)]*)\)\s*</tool_call>"
    matches = re.findall(pattern, text)
    for name, args_str in matches:
        try:
            args = json.loads(f"{{{args_str}}}") if args_str else {}
            tool_calls.append({"name": name, "args": args})
        except json.JSONDecodeError:
            tool_calls.append({"name": name, "args": {}, "parse_error": args_str})
    return tool_calls

def evaluate_on_toolbench(model, tokenizer, num_samples=100, max_new_tokens=512):
    try:
        eval_data = load_dataset("younissk/tool-calling-mix", split="test")
        eval_data = eval_data.select(range(min(num_samples, len(eval_data))))
    except Exception as e:
        print(f"Could not load ToolBench eval set: {e}")
        print("Skipping ToolBench evaluation.")
        return {}

    results = {"correct": 0, "total": 0, "errors": 0}

    for example in tqdm(eval_data, desc="Evaluating"):
        try:
            messages = json.loads(example.get("messages", "[]"))
            if not messages:
                continue
            prompt_messages = []
            for msg in messages[:-1]:
                role = msg.get("role", "user")
                content = msg.get("content", "")
                if content:
                    prompt_messages.append({"role": role, "content": content})
            text = tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            inputs = tokenizer.tokenizer(text, return_tensors="pt").to(model.device)
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
            )
            response = tokenizer.tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

            expected_tools = messages[-1].get("tool_calls", [])
            actual_tools = extract_tool_calls(response)

            if expected_tools and actual_tools:
                expected_names = {t.get("function", {}).get("name", "") for t in expected_tools}
                actual_names = {t["name"] for t in actual_tools}
                if expected_names == actual_names:
                    results["correct"] += 1
                results["total"] += 1
        except Exception:
            results["errors"] += 1

    if results["total"] > 0:
        results["accuracy"] = results["correct"] / results["total"]
    print(f"
ToolBench Results ({num_samples} samples):")
    print(f"  Accuracy: {results.get("accuracy", 0):.2%}")
    print(f"  Correct: {results["correct"]} / {results["total"]}")
    print(f"  Errors: {results["errors"]}")
    return results

print("Running ToolBench evaluation...")
FastModel.for_inference(model)
toolbench_results = evaluate_on_toolbench(model, tokenizer, num_samples=100)

if wandb_key and toolbench_results:
    import wandb
    wandb.log({"eval/toolbench_accuracy": toolbench_results.get("accuracy", 0)})
    wandb.log({"eval/toolbench_correct": toolbench_results["correct"]})
    wandb.log({"eval/toolbench_total": toolbench_results["total"]})


## Push LoRA Adapter to HuggingFace Hub

Upload the trained LoRA adapter so it can be loaded and merged anywhere.

In [ ]:
# ============================================================
# PUSH LORA ADAPTER TO HUGGINGFACE HUB
# ============================================================
if hf_token:
    from huggingface_hub import whoami
    user_info = whoami(token=hf_token)
    username = user_info["name"]
    lora_repo_id = f"{username}/gemma-4-e4b-hermes-agent-reasoning-lora"
    print(f"Authenticated as: {username}")
    print(f"Uploading LoRA adapter to {lora_repo_id}...")
    model.push_to_hub(lora_repo_id, token=hf_token)
    tokenizer.tokenizer.push_to_hub(lora_repo_id, token=hf_token)
    print(f"LoRA pushed: https://huggingface.co/{lora_repo_id}")
else:
    print("HF_TOKEN not set. Skipping push to HuggingFace Hub.")


## Save Model

Export trained artifacts:
1. **LoRA Adapter** - lightweight (~100MB), already pushed to HF Hub
2. **Merged 16-bit** - full model (~8GB), may fail on Colab disk limit

In [ ]:
# ============================================================
# SAVE MERGED 16-BIT MODEL
# ============================================================
merged_save_path = f"{output_directory}gemma4-e4b-hermes-merged"
try:
    model.save_pretrained_merged(
        merged_save_path,
        tokenizer,
        save_method = "merged_16bit",
    )
    print(f"Merged 16bit model saved to: {merged_save_path}")
except Exception as e:
    print(f"Failed to save merged model: {e}")
    print("The LoRA adapter on HF Hub is the primary artifact.")


## GGUF Export (Optional)

Attempts to export the model to GGUF Q4_K_M format for llama.cpp/Ollama inference.

**Note:** Requires significant disk space. May fail on Colab.

In [ ]:
# ============================================================
# TRY DIRECT GGUF EXPORT
# ============================================================
gguf_path = f"{output_directory}gemma4-e4b-hermes-q4km"
GGUF_SUCCESS = False
try:
    model.save_pretrained_gguf(
        gguf_path,
        tokenizer,
        quantization_method = "q4_k_m",
    )
    print(f"GGUF Q4_K_M saved to: {gguf_path}")
    GGUF_SUCCESS = True
except Exception as e:
    print(f"Direct GGUF export failed: {e}")
    print("Skipping GGUF export.")


---

## Summary

| Artifact | Location |
|----------|----------|
| LoRA Adapter | https://huggingface.co/<username>/gemma-4-e4b-hermes-agent-reasoning-lora |
| Merged 16-bit (if saved) | /content/gemma4-e4b-hermes-merged |
| GGUF Q4_K_M (if saved) | /content/gemma4-e4b-hermes-q4km |
| W&B Run | https://wandb.ai/<username>/gemma4-e4b-hermes-agent-reasoning |

### Next Steps
- Load the LoRA adapter in TinyCUA: FastModel.from_pretrained(..., adapter="<repo>")
- Test tool-call prompts against the fine-tuned model
- Run full training (1 epoch) for production quality
